# GigBid: tool-granularity study (R&D question A) on a free Kaggle GPU

A short run (about 20 minutes). 40 tasks with a known correct tool and arguments are given to the model twice:
once with the 17 fine-grained MCP tools, once with 3 coarse tools that take an `action` and free `params`.
It reports tool-selection accuracy, argument accuracy and tokens per task. The hypothesis is written at the top of
`evals/runners/tool_study.py`, before the first run.

It writes `evals/results/tool_study.json` and prints it at the end with its SHA-256, so the result can be copied
from the log if the download is not possible.

Before you run it, in the right-hand panel set **Accelerator: GPU T4 x2** and **Internet: On**.


In [ ]:
# 1. Get the code
REPO = "https://github.com/aimennnnn/gigbid.git"
!rm -rf /kaggle/working/gigbid
!git clone --depth 1 $REPO /kaggle/working/gigbid
%cd /kaggle/working/gigbid
!git log --oneline -1

In [ ]:
# 2. Install the project (uv reads pyproject.toml and uv.lock, the same versions as on the laptop)
!pip install -q uv
!uv sync --frozen

In [ ]:
# 3. Install Ollama, start it in the background and download the model (about 14 GB)
import os
import subprocess
import time
!apt-get install -y -qq pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
# a bigger context window than Ollama's default, so long prompts (job + evidence + schema) are not cut off
server = subprocess.Popen(["ollama", "serve"], env=os.environ | {"OLLAMA_CONTEXT_LENGTH": "16384"},
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for attempt in range(1, 11):          # the model registry sometimes answers "503": wait and try again
    time.sleep(15)
    pull = subprocess.run(["ollama", "pull", "gpt-oss:20b"], capture_output=True, text=True)
    if pull.returncode == 0:
        break
    print(f"download attempt {attempt} failed: {pull.stderr.strip()[-200:]}")
models = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
print(models)
assert "gpt-oss:20b" in models, "the model was not downloaded, so the LLM steps cannot run"

In [ ]:
# 4. Point GigBid at the local model. These names are read by agent/llm.py (see agent/config.yaml, provider "ollama").
os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"
os.environ["GIGBID_PROVIDER"] = "ollama"

# the hardware goes into the results folder, because the report must state it for local models
!mkdir -p evals/results
!nvidia-smi --query-gpu=name,memory.total --format=csv | tee evals/results/kaggle_hardware.txt
!nproc | xargs echo "CPU cores:" | tee -a evals/results/kaggle_hardware.txt

In [ ]:
# 5. Quick check: one real job through the bid graph. It must end without a model error, or the notebook stops here.
check = subprocess.run(["uv", "run", "python", "-m", "agent.demo"], capture_output=True, text=True)
print(check.stdout[-4000:])
print(check.stderr[-2000:])
assert "status=" in check.stdout and "ERRORS" not in check.stdout, "the local model did not get one job through the graph"

In [ ]:
# 6. The study: 40 tasks x 2 tool sets = 80 model calls.
!uv run python -m evals.runners.tool_study


In [ ]:
# 7. Print the result file and its hash, then pack it for download.
import hashlib
import pathlib
raw = pathlib.Path("evals/results/tool_study.json").read_bytes()
print("SHA256", hashlib.sha256(raw).hexdigest(), "BYTES", len(raw))
print("-----BEGIN tool_study.json-----")
print(raw.decode("utf-8"))
print("-----END tool_study.json-----")
!cd /kaggle/working/gigbid && zip -q /kaggle/working/gigbid_tool_study.zip evals/results/tool_study.json
print("Download /kaggle/working/gigbid_tool_study.zip and unzip it over your gigbid folder.")
